### SHAP VALUE

https://github.com/shap/shap

In [1]:
import shap

# Explicando o modelo com SHAP
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test)

# Gráfico de valores SHAP
shap.summary_plot(shap_values, X_test)


NameError: name 'final_model' is not defined

### IMPORTÂNCIA POR PERMUTAÇÃO

https://scikit-learn.org/1.1/modules/permutation_importance.html

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

# Importância por permutação
result = permutation_importance(final_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

# Ordenando os resultados
sorted_idx = result.importances_mean.argsort()

# Plotando
plt.figure(figsize=(10, 6))
plt.barh(X_test.columns[sorted_idx], result.importances_mean[sorted_idx], color='skyblue')
plt.xlabel("Importância por Permutação")
plt.title("Importância das Features via Permutação")
plt.tight_layout()
plt.show()


### LIME

https://lime-ml.readthedocs.io/en/latest

In [1]:
pip install lime

import lime
import lime.lime_tabular
import numpy as np

# Criando o explicador LIME
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_final.values,        # Usa X_train_final
    feature_names=X_train_final.columns.tolist(),
    class_names=["Não Inadimplente", "Inadimplente"],
    mode="classification"
)

# Explicando algumas previsões
for i in [3, 10, 25, 50, 99]:
    exp = explainer.explain_instance(
        data_row=X_test.iloc[i].values,         # Pega a linha como array
        predict_fn=final_model.predict_proba    # Usa final_model
    )
    exp.show_in_notebook(show_table=True)


### METRICS FAIRNESS


https://fairlearn.org/v0.12/user_guide/assessment/perform_fairness_assessment.html

In [ ]:
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate, demographic_parity_difference, equalized_odds_difference

# Fairness por grupo sensível (Sexo)
sensitive_feature = X_test['SEX'].map({1: 'Masculino', 2: 'Feminino'})

# Métricas de fairness
fairness_metrics = {
    'Demographic Parity (Selection Rate)': lambda y_true, y_pred: (y_pred==1).mean(),
    'True Positive Rate': true_positive_rate,
    'False Positive Rate': false_positive_rate,
}
mf = MetricFrame(metrics=fairness_metrics, y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

### MDE

MDE (Mean Direct Effect) é o efeito médio que o modelo sofre quando você intervém e altera diretamente o atributo protegido, mantendo todo o resto igual.
Em outras palavras: mede quanto a predição muda somente por trocar o valor do atributo sensível (ex.: SEX 1→2)

https://www.catalyzex.com/paper/marrying-fairness-and-explainability-in

In [ ]:
mport numpy as np
import pandas as pd

# Função para calcular MDE
def compute_mde_interventional(model, X, protected_attr: str, z0, z1, use_proba=True):
    X0 = X.copy()
    X1 = X.copy()
    X0[protected_attr] = z0
    X1[protected_attr] = z1
    if hasattr(model, "predict_proba") and use_proba:
        p0 = model.predict_proba(X0)[:, 1]
        p1 = model.predict_proba(X1)[:, 1]
    else:
        p0 = model.predict(X0).astype(float)
        p1 = model.predict(X1).astype(float)
    delta = p1 - p0
    mde = float(np.mean(delta))
    return mde, delta

# IC bootstrap
def ci_bootstrap_mean(values, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(values)
    boots = [np.mean(values[rng.integers(0, n, n)]) for _ in range(n_boot)]
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return float(lo), float(hi)

# MDE por grupo
def mde_by_group(model, X, group_col, protected_attr, z0, z1):
    out = []
    for g in sorted(X[group_col].dropna().unique()):
        mask = X[group_col] == g
        if mask.sum() < 20:
            continue
        mde_g, _ = compute_mde_interventional(model, X[mask], protected_attr, z0, z1)
        out.append((g, mde_g, mask.sum()))
    return pd.DataFrame(out, columns=[group_col, "MDE", "n"]).sort_values("MDE")

# MDE global para SEX
mde_sex, delta_sex = compute_mde_interventional(final_model, X_test, "SEX", z0=1, z1=2)
lo, hi = ci_bootstrap_mean(delta_sex, n_boot=1000)
print(f"MDE (SEX: 1→2) = {mde_sex:.6f}")
print(f"IC 95% para MDE: [{lo:.6f}, {hi:.6f}]")

# MDE por EDUCATION
print("\nMDE por EDUCATION:")
print(mde_by_group(final_model, X_test, "EDUCATION", "SEX", 1, 2))

# MDE por MARRIAGE
print("\nMDE por MARRIAGE:")
print(mde_by_group(final_model, X_test, "MARRIAGE", "SEX", 1, 2))

### FGAP

Fairness Gap (FGAP) — diferença observada entre métricas de desempenho ou taxas de seleção para diferentes grupos protegidos de um modelo (por exemplo, diferença na taxa de aprovação entre grupos).
Ou seja: mede o “gap” real na saída do modelo entre os grupos.

https://www.sciencedirect.com/science/article/abs/pii/S1386505624001977